In [16]:
from dataclasses import dataclass
from typing import Tuple

# =========================================================
# INITIALIZATION / GLOBAL SETTINGS
# Change everything important here
# =========================================================
@dataclass
class CABConfig:
    # Forecast setup
    h: int = 21                  # forecast horizon
    L: int = 100                 # sequence length

    # Assets
    asset_order: Tuple[str, ...] = (
        "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
        "US_REIT", "INTL_REIT", "GOLD", "BTC"
    )

    # Data
    data_dir: str = "../proxy"
    input_lag: int = 21          # which lag block to use as D_t

    # Time split
    train_end_date: str = "2020-12-31"
    test_start_date: str = "2021-01-01"
    test_end_date: str = "2025-12-31"

    # Scaling
    scale_inputs: bool = True
    scale_targets: bool = False
    eps: float = 1e-12

    # Training hyperparameters
    batch_size: int = 32
    epochs_initial: int = 50
    epochs_refit: int = 5
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5

    # CNN / CAB architecture hyperparameters
    kernel_size: int = 5
    hidden_dim: int = 128
    num_attention_heads: int = 8
    num_lstm_layers: int = 2
    dropout: float = 0.1

    # Recalibration
    refit_every: int = 21
    use_warm_start: bool = True


CFG = CABConfig()

# ---------------------------------------------------------
# Derived settings
# ---------------------------------------------------------
COV_DATA_PATH = f"{CFG.data_dir}/realized_cov_target_h{CFG.h}_lag_5_10_21_63.csv"
N_ASSETS = len(CFG.asset_order)
INPUT_PREFIX = f"lag{CFG.input_lag}"
TARGET_PREFIX = "target"
RUN_NAME = f"cab_h{CFG.h}_L{CFG.L}_lag{CFG.input_lag}"

print("CAB configuration loaded.")
print("Run name             :", RUN_NAME)
print("Covariance data file :", COV_DATA_PATH)
print("Horizon h            :", CFG.h)
print("Sequence length L    :", CFG.L)
print("Input lag            :", CFG.input_lag)
print("Kernel size          :", CFG.kernel_size)
print("Train end            :", CFG.train_end_date)
print("Test period          :", CFG.test_start_date, "->", CFG.test_end_date)
print("Assets               :", N_ASSETS)
print("Asset order          :", CFG.asset_order)

CAB configuration loaded.
Run name             : cab_h21_L100_lag21
Covariance data file : ../proxy/realized_cov_target_h21_lag_5_10_21_63.csv
Horizon h            : 21
Sequence length L    : 100
Input lag            : 21
Kernel size          : 5
Train end            : 2020-12-31
Test period          : 2021-01-01 -> 2025-12-31
Assets               : 8
Asset order          : ('US_EQUITY', 'INTL_EQUITY', 'US_BONDS', 'INTL_BONDS', 'US_REIT', 'INTL_REIT', 'GOLD', 'BTC')


In [17]:
import torch
import torch.nn as nn

In [18]:
def reshape_for_conv3d(x: torch.Tensor, cfg) -> torch.Tensor:
    """
    Convert CAB input from shape (B, L, N, N) to (B, 1, L, N, N).

    Parameters
    ----------
    x : torch.Tensor
        Input tensor with shape (batch_size, sequence_length, n_assets, n_assets).
    cfg : CABConfig
        Global configuration object.

    Returns
    -------
    torch.Tensor
        Reshaped tensor with shape (batch_size, 1, sequence_length, n_assets, n_assets).
    """
    if x.ndim != 4:
        raise ValueError(f"Expected x.ndim == 4, got {x.ndim}")

    B, L, N1, N2 = x.shape

    if L != cfg.L:
        raise ValueError(f"Expected sequence length {cfg.L}, got {L}")

    if N1 != len(cfg.asset_order) or N2 != len(cfg.asset_order):
        raise ValueError(
            f"Expected matrix size ({len(cfg.asset_order)}, {len(cfg.asset_order)}), "
            f"got ({N1}, {N2})"
        )

    return x.unsqueeze(1)

In [19]:
def flatten_conv_output_for_lstm(x_conv: torch.Tensor) -> torch.Tensor:
    """
    Convert CNN output from shape (B, C, L, N, N) to (B, L, C*N*N).

    Parameters
    ----------
    x_conv : torch.Tensor
        CNN output tensor.

    Returns
    -------
    torch.Tensor
        Flattened sequence tensor suitable for an LSTM.
    """
    if x_conv.ndim != 5:
        raise ValueError(f"Expected x_conv.ndim == 5, got {x_conv.ndim}")

    B, C, L, N1, N2 = x_conv.shape

    # Move time dimension to second axis: (B, L, C, N, N)
    x_conv = x_conv.permute(0, 2, 1, 3, 4).contiguous()

    # Flatten each time step
    x_flat = x_conv.view(B, L, C * N1 * N2)

    return x_flat

In [20]:
class CAB3DCNNBlock(nn.Module):
    """
    First CAB stage: 3D convolution over a sequence of covariance matrices.

    Input
    -----
    x : torch.Tensor
        Shape (B, L, N, N)

    Internal Conv3D input
    ---------------------
        Shape (B, 1, L, N, N)

    Outputs
    -------
    x_conv : torch.Tensor
        Shape (B, 1, L, N, N) if out_channels = 1
    x_flat : torch.Tensor
        Shape (B, L, N*N) if out_channels = 1
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        padding = cfg.kernel_size // 2

        self.conv3d = nn.Conv3d(
            in_channels=1,
            out_channels=1,
            kernel_size=(cfg.kernel_size, cfg.kernel_size, cfg.kernel_size),
            stride=1,
            padding=(padding, padding, padding)
        )

        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor):
        # (B, L, N, N) -> (B, 1, L, N, N)
        x = reshape_for_conv3d(x, self.cfg)

        # 3D convolution
        x_conv = self.conv3d(x)

        # Nonlinearity
        x_conv = self.activation(x_conv)

        # Flatten for later recurrent layer
        x_flat = flatten_conv_output_for_lstm(x_conv)

        return x_conv, x_flat

In [22]:
CNN_FEATURE_DIM = N_ASSETS * N_ASSETS   # valid for out_channels = 1

print("CNN feature dimension :", CNN_FEATURE_DIM)
print("BiLSTM hidden dim     :", CFG.hidden_dim)
print("BiLSTM output dim     :", 2 * CFG.hidden_dim)

CNN feature dimension : 64
BiLSTM hidden dim     : 128
BiLSTM output dim     : 256


In [23]:
def validate_lstm_input(x_seq: torch.Tensor, cfg) -> None:
    """
    Validate that the BiLSTM input has shape (B, L, F).

    Parameters
    ----------
    x_seq : torch.Tensor
        Sequence tensor from the CNN block.
    cfg : CABConfig
        Global configuration object.
    """
    if x_seq.ndim != 3:
        raise ValueError(f"Expected x_seq.ndim == 3, got {x_seq.ndim}")

    B, L, F = x_seq.shape

    if L != cfg.L:
        raise ValueError(f"Expected sequence length {cfg.L}, got {L}")

    if F <= 0:
        raise ValueError(f"Feature dimension must be positive, got {F}")

In [24]:
class CABBiLSTMBlock(nn.Module):
    """
    CAB BiLSTM stage.

    Input
    -----
    x_seq : torch.Tensor
        Shape (B, L, F)

    Output
    ------
    x_lstm : torch.Tensor
        Shape (B, L, 2 * hidden_dim)
    """
    def __init__(self, cfg, input_dim: int):
        super().__init__()
        self.cfg = cfg
        self.input_dim = input_dim

        self.bilstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=cfg.hidden_dim,
            num_layers=cfg.num_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg.dropout if cfg.num_lstm_layers > 1 else 0.0
        )

        self.output_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        validate_lstm_input(x_seq, self.cfg)

        x_lstm, _ = self.bilstm(x_seq)
        x_lstm = self.output_dropout(x_lstm)

        return x_lstm

In [25]:
bilstm_block = CABBiLSTMBlock(
    cfg=CFG,
    input_dim=x_flat.shape[-1]
)

x_lstm = bilstm_block(x_flat)

print("CNN flattened input shape :", x_flat.shape)
print("BiLSTM output shape       :", x_lstm.shape)

CNN flattened input shape : torch.Size([4, 100, 64])
BiLSTM output shape       : torch.Size([4, 100, 256])


In [26]:
def print_bilstm_dimensions(x_in: torch.Tensor, x_out: torch.Tensor, cfg) -> None:
    """
    Print the key BiLSTM dimensions.
    """
    print("BiLSTM input:")
    print("  batch size      :", x_in.shape[0])
    print("  sequence length :", x_in.shape[1])
    print("  input dim       :", x_in.shape[2])

    print("\nBiLSTM output:")
    print("  batch size      :", x_out.shape[0])
    print("  sequence length :", x_out.shape[1])
    print("  output dim      :", x_out.shape[2])

    print("\nExpected output dim =", 2 * cfg.hidden_dim)

print_bilstm_dimensions(x_flat, x_lstm, CFG)

BiLSTM input:
  batch size      : 4
  sequence length : 100
  input dim       : 64

BiLSTM output:
  batch size      : 4
  sequence length : 100
  output dim      : 256

Expected output dim = 256


In [27]:
def validate_attention_config(cfg) -> None:
    """
    Validate that the attention setup is internally consistent.

    In particular, embed_dim = 2 * hidden_dim must be divisible by the
    number of attention heads.
    """
    embed_dim = 2 * cfg.hidden_dim

    if embed_dim % cfg.num_attention_heads != 0:
        raise ValueError(
            f"embed_dim = 2 * hidden_dim = {embed_dim} must be divisible by "
            f"num_attention_heads = {cfg.num_attention_heads}"
        )

In [28]:
def validate_attention_input(x_seq: torch.Tensor, cfg) -> None:
    """
    Validate that the attention input has shape (B, L, F),
    where F = 2 * hidden_dim.

    Parameters
    ----------
    x_seq : torch.Tensor
        Sequence tensor from the BiLSTM block.
    cfg : CABConfig
        Global configuration object.
    """
    if x_seq.ndim != 3:
        raise ValueError(f"Expected x_seq.ndim == 3, got {x_seq.ndim}")

    B, L, F = x_seq.shape

    if L != cfg.L:
        raise ValueError(f"Expected sequence length {cfg.L}, got {L}")

    expected_F = 2 * cfg.hidden_dim
    if F != expected_F:
        raise ValueError(f"Expected feature dimension {expected_F}, got {F}")

In [29]:
def mean_pool_sequence(x_seq: torch.Tensor) -> torch.Tensor:
    """
    Mean-pool across the sequence dimension.

    Parameters
    ----------
    x_seq : torch.Tensor
        Tensor of shape (B, L, F)

    Returns
    -------
    torch.Tensor
        Tensor of shape (B, F)
    """
    if x_seq.ndim != 3:
        raise ValueError(f"Expected x_seq.ndim == 3, got {x_seq.ndim}")

    return x_seq.mean(dim=1)

In [31]:
class CABAttentionBlock(nn.Module):
    """
    CAB multi-head self-attention block.

    Input
    -----
    x_seq : torch.Tensor
        Shape (B, L, 2 * hidden_dim)

    Outputs
    -------
    x_attn : torch.Tensor
        Attended sequence, shape (B, L, 2 * hidden_dim)

    a_mean : torch.Tensor
        Mean-pooled context vector, shape (B, 2 * hidden_dim)

    attn_weights : torch.Tensor
        Attention weights returned by PyTorch.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed_dim = 2 * cfg.hidden_dim

        validate_attention_config(cfg)

        self.attention = nn.MultiheadAttention(
            embed_dim=self.embed_dim,
            num_heads=cfg.num_attention_heads,
            dropout=cfg.dropout,
            batch_first=True
        )

        self.output_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x_seq: torch.Tensor):
        validate_attention_input(x_seq, self.cfg)

        x_attn, attn_weights = self.attention(
            query=x_seq,
            key=x_seq,
            value=x_seq
        )

        x_attn = self.output_dropout(x_attn)
        a_mean = mean_pool_sequence(x_attn)

        return x_attn, a_mean, attn_weights

In [32]:
attention_block = CABAttentionBlock(CFG)

x_attn, a_mean, attn_weights = attention_block(x_lstm)

print("BiLSTM input shape      :", x_lstm.shape)
print("Attention output shape  :", x_attn.shape)
print("Mean pooled shape       :", a_mean.shape)
print("Attention weights shape :", attn_weights.shape)

BiLSTM input shape      : torch.Size([4, 100, 256])
Attention output shape  : torch.Size([4, 100, 256])
Mean pooled shape       : torch.Size([4, 256])
Attention weights shape : torch.Size([4, 100, 100])


In [34]:
def print_attention_dimensions(
    x_in: torch.Tensor,
    x_attn: torch.Tensor,
    a_mean: torch.Tensor,
    attn_weights: torch.Tensor,
    cfg
) -> None:
    """
    Print the key dimensions of the attention block.
    """
    print("Attention input:")
    print("  batch size      :", x_in.shape[0])
    print("  sequence length :", x_in.shape[1])
    print("  feature dim     :", x_in.shape[2])

    print("\nAttention output:")
    print("  batch size      :", x_attn.shape[0])
    print("  sequence length :", x_attn.shape[1])
    print("  feature dim     :", x_attn.shape[2])

    print("\nMean pooled context:")
    print("  batch size      :", a_mean.shape[0])
    print("  feature dim     :", a_mean.shape[1])

    print("\nExpected pooled dim =", 2 * cfg.hidden_dim)
    print("Attention weights shape:", attn_weights.shape)
    
print_attention_dimensions(x_lstm, x_attn, a_mean, attn_weights, CFG)

Attention input:
  batch size      : 4
  sequence length : 100
  feature dim     : 256

Attention output:
  batch size      : 4
  sequence length : 100
  feature dim     : 256

Mean pooled context:
  batch size      : 4
  feature dim     : 256

Expected pooled dim = 256
Attention weights shape: torch.Size([4, 100, 100])


In [35]:
def validate_head_input(a_mean: torch.Tensor, cfg) -> None:
    """
    Validate that the forecast head input has shape (B, 2 * hidden_dim).
    """
    if a_mean.ndim != 2:
        raise ValueError(f"Expected a_mean.ndim == 2, got {a_mean.ndim}")

    B, F = a_mean.shape
    expected_F = 2 * cfg.hidden_dim

    if F != expected_F:
        raise ValueError(f"Expected feature dimension {expected_F}, got {F}")

In [36]:
def flat_to_matrix(y_flat: torch.Tensor, cfg) -> torch.Tensor:
    """
    Convert flat forecast output from shape (B, N*N) to (B, N, N).
    """
    if y_flat.ndim != 2:
        raise ValueError(f"Expected y_flat.ndim == 2, got {y_flat.ndim}")

    B, D = y_flat.shape
    n_assets = len(cfg.asset_order)
    expected_D = n_assets * n_assets

    if D != expected_D:
        raise ValueError(f"Expected flat dimension {expected_D}, got {D}")

    return y_flat.view(B, n_assets, n_assets)

In [37]:
def enforce_symmetry(Y: torch.Tensor) -> torch.Tensor:
    """
    Enforce symmetry on a batch of square matrices.

    Parameters
    ----------
    Y : torch.Tensor
        Shape (B, N, N)

    Returns
    -------
    torch.Tensor
        Symmetrized matrices, shape (B, N, N)
    """
    if Y.ndim != 3:
        raise ValueError(f"Expected Y.ndim == 3, got {Y.ndim}")

    return 0.5 * (Y + Y.transpose(-1, -2))

In [38]:
def enforce_psd(Y_sym: torch.Tensor) -> torch.Tensor:
    """
    Enforce positive semi-definiteness by clipping negative eigenvalues to zero.

    Parameters
    ----------
    Y_sym : torch.Tensor
        Symmetric matrices, shape (B, N, N)

    Returns
    -------
    torch.Tensor
        PSD matrices, shape (B, N, N)
    """
    if Y_sym.ndim != 3:
        raise ValueError(f"Expected Y_sym.ndim == 3, got {Y_sym.ndim}")

    eigvals, eigvecs = torch.linalg.eigh(Y_sym)
    eigvals_clipped = torch.clamp(eigvals, min=0.0)

    Y_psd = eigvecs @ torch.diag_embed(eigvals_clipped) @ eigvecs.transpose(-1, -2)
    return Y_psd

In [39]:
def inspect_matrix_batch(Y_raw: torch.Tensor, Y_sym: torch.Tensor, Y_psd: torch.Tensor) -> None:
    """
    Print simple diagnostics for a batch of matrices.
    """
    raw_sym_error = torch.max(torch.abs(Y_raw - Y_raw.transpose(-1, -2))).item()
    sym_sym_error = torch.max(torch.abs(Y_sym - Y_sym.transpose(-1, -2))).item()
    psd_sym_error = torch.max(torch.abs(Y_psd - Y_psd.transpose(-1, -2))).item()

    min_eig_sym = torch.min(torch.linalg.eigvalsh(Y_sym)).item()
    min_eig_psd = torch.min(torch.linalg.eigvalsh(Y_psd)).item()

    print("Max asymmetry raw :", raw_sym_error)
    print("Max asymmetry sym :", sym_sym_error)
    print("Max asymmetry psd :", psd_sym_error)
    print("Min eigenvalue sym:", min_eig_sym)
    print("Min eigenvalue psd:", min_eig_psd)

In [40]:
class CABForecastHead(nn.Module):
    """
    Final CAB forecast head.

    Input
    -----
    a_mean : torch.Tensor
        Shape (B, 2 * hidden_dim)

    Outputs
    -------
    y_flat : torch.Tensor
        Shape (B, N*N)

    Y_raw : torch.Tensor
        Shape (B, N, N)
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.n_assets = len(cfg.asset_order)
        self.output_dim = self.n_assets * self.n_assets

        self.fc = nn.Linear(2 * cfg.hidden_dim, self.output_dim)

    def forward(self, a_mean: torch.Tensor):
        validate_head_input(a_mean, self.cfg)

        y_flat = self.fc(a_mean)
        Y_raw = flat_to_matrix(y_flat, self.cfg)

        return y_flat, Y_raw

In [41]:
forecast_head = CABForecastHead(CFG)

y_flat, Y_raw = forecast_head(a_mean)

print("Mean pooled input shape :", a_mean.shape)
print("Flat output shape       :", y_flat.shape)
print("Raw matrix shape        :", Y_raw.shape)

Mean pooled input shape : torch.Size([4, 256])
Flat output shape       : torch.Size([4, 64])
Raw matrix shape        : torch.Size([4, 8, 8])


In [42]:
Y_sym = enforce_symmetry(Y_raw)
Y_psd = enforce_psd(Y_sym)

print("Raw matrix shape :", Y_raw.shape)
print("Sym matrix shape :", Y_sym.shape)
print("PSD matrix shape :", Y_psd.shape)

inspect_matrix_batch(Y_raw, Y_sym, Y_psd)

Raw matrix shape : torch.Size([4, 8, 8])
Sym matrix shape : torch.Size([4, 8, 8])
PSD matrix shape : torch.Size([4, 8, 8])
Max asymmetry raw : 0.1106889396905899
Max asymmetry sym : 0.0
Max asymmetry psd : 3.725290298461914e-09
Min eigenvalue sym: -0.10089129209518433
Min eigenvalue psd: -1.710724184533774e-08


In [44]:
def postprocess_covariance_forecast(Y_raw: torch.Tensor) -> torch.Tensor:
    """
    Post-process raw covariance forecasts:
    1. enforce symmetry
    2. enforce PSD
    """
    Y_sym = enforce_symmetry(Y_raw)
    Y_psd = enforce_psd(Y_sym)
    return Y_psd

Y_final = postprocess_covariance_forecast(Y_raw)
print("Final processed forecast shape:", Y_final.shape)

Final processed forecast shape: torch.Size([4, 8, 8])


In [45]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

# =========================================================
# DEVICE
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# =========================================================
# FULL CAB MODEL
# Assumes these already exist from earlier cells:
# - CAB3DCNNBlock
# - CABBiLSTMBlock
# - CABAttentionBlock
# - CABForecastHead
# - enforce_symmetry
# =========================================================
class CABModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        self.cnn_block = CAB3DCNNBlock(cfg)

        cnn_feature_dim = len(cfg.asset_order) * len(cfg.asset_order)

        self.bilstm_block = CABBiLSTMBlock(
            cfg=cfg,
            input_dim=cnn_feature_dim
        )

        self.attention_block = CABAttentionBlock(cfg)
        self.forecast_head = CABForecastHead(cfg)

    def forward(self, x: torch.Tensor):
        x_conv, x_flat = self.cnn_block(x)
        x_lstm = self.bilstm_block(x_flat)
        x_attn, a_mean, attn_weights = self.attention_block(x_lstm)
        y_flat, y_raw = self.forecast_head(a_mean)
        y_sym = enforce_symmetry(y_raw)

        return {
            "x_conv": x_conv,
            "x_flat": x_flat,
            "x_lstm": x_lstm,
            "x_attn": x_attn,
            "a_mean": a_mean,
            "attn_weights": attn_weights,
            "y_flat": y_flat,
            "y_raw": y_raw,
            "y_sym": y_sym
        }

# =========================================================
# TRAINING HELPERS
# =========================================================
def make_dataloader(X, Y, batch_size, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    Y_tensor = torch.tensor(Y, dtype=torch.float32)

    dataset = TensorDataset(X_tensor, Y_tensor)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False
    )
    return loader


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    n_samples = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for x_batch, y_batch in pbar:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        out = model(x_batch)
        y_pred = out["y_sym"]

        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()

        batch_size = x_batch.size(0)
        running_loss += loss.item() * batch_size
        n_samples += batch_size

        pbar.set_postfix({"batch_loss": f"{loss.item():.6f}"})

    epoch_loss = running_loss / n_samples
    return epoch_loss


def fit_cab_model(X_train, Y_train, cfg, device):
    model = CABModel(cfg).to(device)

    train_loader = make_dataloader(
        X=X_train,
        Y=Y_train,
        batch_size=cfg.batch_size,
        shuffle=True
    )

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay
    )

    history = []

    print("Training samples :", len(X_train))
    print("Batch size       :", cfg.batch_size)
    print("Epochs           :", cfg.epochs_initial)

    for epoch in range(1, cfg.epochs_initial + 1):
        epoch_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device
        )

        history.append(epoch_loss)
        print(f"Epoch {epoch:03d}/{cfg.epochs_initial:03d} | train_loss = {epoch_loss:.6f}")

    return model, history

Using device: cpu


In [48]:
# =========================================================
# ONE-CELL REAL-DATA TRAINING RUN
# Assumes these already exist from earlier cells:
# - CFG
# - COV_DATA_PATH
# - DEVICE
# - fit_cab_model(...)
# =========================================================
import numpy as np
import pandas as pd

# -----------------------------
# 1) Load covariance dataset
# -----------------------------
df_cov = pd.read_csv(
    COV_DATA_PATH,
    parse_dates=["Date", "target_start", "target_end"]
).sort_values("Date").reset_index(drop=True)

print("Loaded covariance data:", COV_DATA_PATH)
print("Rows:", len(df_cov))
print("Date range:", df_cov["Date"].min().date(), "->", df_cov["Date"].max().date())

# -----------------------------
# 2) Helper: row -> covariance matrix
# -----------------------------
def row_to_cov_matrix(row: pd.Series, prefix: str, asset_order) -> np.ndarray:
    n = len(asset_order)
    M = np.zeros((n, n), dtype=float)

    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            col = f"{prefix}_cov_{a}__{b}"
            if col not in row.index:
                raise ValueError(f"Missing column: {col}")
            M[i, j] = row[col]

    return M

# -----------------------------
# 3) Build daily matrix streams
# D_t = chosen lagged covariance
# S_t = future target covariance
# -----------------------------
input_prefix = f"lag{CFG.input_lag}"
target_prefix = "target"

D_list = []
S_list = []

for _, row in df_cov.iterrows():
    D_list.append(row_to_cov_matrix(row, prefix=input_prefix, asset_order=CFG.asset_order))
    S_list.append(row_to_cov_matrix(row, prefix=target_prefix, asset_order=CFG.asset_order))

D_all = np.asarray(D_list, dtype=float)
S_all = np.asarray(S_list, dtype=float)

print("D_all shape:", D_all.shape)
print("S_all shape:", S_all.shape)

# -----------------------------
# 4) Build sliding-window samples
# X_t = [D_{t-L+1}, ..., D_t]
# Y_t = S_t
# -----------------------------
X_list = []
Y_list = []
sample_dates = []
sample_target_end = []

for end_idx in range(CFG.L - 1, len(df_cov)):
    X_seq = D_all[end_idx - CFG.L + 1 : end_idx + 1]   # (L, N, N)
    Y_mat = S_all[end_idx]                             # (N, N)

    X_list.append(X_seq)
    Y_list.append(Y_mat)
    sample_dates.append(df_cov.loc[end_idx, "Date"])
    sample_target_end.append(df_cov.loc[end_idx, "target_end"])

X_all = np.asarray(X_list, dtype=float)
Y_all = np.asarray(Y_list, dtype=float)
sample_dates = pd.to_datetime(pd.Series(sample_dates))
sample_target_end = pd.to_datetime(pd.Series(sample_target_end))

print("X_all shape:", X_all.shape)
print("Y_all shape:", Y_all.shape)

# -----------------------------
# 5) Keep only training samples
# Target window must be fully known by train_end_date
# -----------------------------
train_mask = sample_target_end <= pd.Timestamp(CFG.train_end_date)

X_train = X_all[train_mask.to_numpy()]
Y_train = Y_all[train_mask.to_numpy()]
dates_train = sample_dates[train_mask].reset_index(drop=True)

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)

if len(dates_train) == 0:
    raise ValueError("No training samples found. Check CFG.L, CFG.h, and CFG.train_end_date.")

print("Training prediction dates:", dates_train.min().date(), "->", dates_train.max().date())

# -----------------------------
# 6) Train the model
# -----------------------------
cab_model, train_history = fit_cab_model(
    X_train=X_train,
    Y_train=Y_train,
    cfg=CFG,
    device=DEVICE
)

print("Real-data training done.")
print("Final training loss:", train_history[-1])
print("Best training loss :", min(train_history))

Loaded covariance data: ../proxy/realized_cov_target_h21_lag_5_10_21_63.csv
Rows: 2121
Date range: 2017-06-27 -> 2025-12-02
D_all shape: (2121, 8, 8)
S_all shape: (2121, 8, 8)
X_all shape: (2022, 100, 8, 8)
Y_all shape: (2022, 8, 8)
X_train shape: (767, 100, 8, 8)
Y_train shape: (767, 8, 8)
Training prediction dates: 2017-11-15 -> 2020-12-02
Training samples : 767
Batch size       : 32
Epochs           : 50


Training:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 001/050 | train_loss = 0.000072


Training:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 002/050 | train_loss = 0.000002


...

Epoch 048/050 | train_loss = 0.000000


Training:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 049/050 | train_loss = 0.000000


Training:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 050/050 | train_loss = 0.000000
Real-data training done.
Final training loss: 2.4535017933987067e-07
Best training loss : 2.3130845634416313e-07


In [52]:
import torch
import os


MODEL_PATH = f"cab_model_h{CFG.h}.pt"

torch.save(cab_model.state_dict(), MODEL_PATH)

print("Model saved to:", MODEL_PATH)

Model saved to: cab_model_h21.pt


Testing 

In [53]:
import numpy as np
import pandas as pd

# =========================================================
# BUILD FULL TEST SET
# =========================================================
df_cov = pd.read_csv(
    COV_DATA_PATH,
    parse_dates=["Date", "target_start", "target_end"]
).sort_values("Date").reset_index(drop=True)

def row_to_cov_matrix(row: pd.Series, prefix: str, asset_order) -> np.ndarray:
    n = len(asset_order)
    M = np.zeros((n, n), dtype=float)

    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            col = f"{prefix}_cov_{a}__{b}"
            M[i, j] = row[col]

    return M

input_prefix = f"lag{CFG.input_lag}"
target_prefix = "target"

D_list = []
S_list = []

for _, row in df_cov.iterrows():
    D_list.append(row_to_cov_matrix(row, prefix=input_prefix, asset_order=CFG.asset_order))
    S_list.append(row_to_cov_matrix(row, prefix=target_prefix, asset_order=CFG.asset_order))

D_all = np.asarray(D_list, dtype=float)
S_all = np.asarray(S_list, dtype=float)

X_list = []
Y_list = []
D_current_list = []
sample_dates = []

for end_idx in range(CFG.L - 1, len(df_cov)):
    X_seq = D_all[end_idx - CFG.L + 1 : end_idx + 1]
    Y_mat = S_all[end_idx]
    D_t = D_all[end_idx]

    X_list.append(X_seq)
    Y_list.append(Y_mat)
    D_current_list.append(D_t)
    sample_dates.append(df_cov.loc[end_idx, "Date"])

X_all = np.asarray(X_list, dtype=float)
Y_all = np.asarray(Y_list, dtype=float)
D_current_all = np.asarray(D_current_list, dtype=float)
sample_dates = pd.to_datetime(pd.Series(sample_dates))

test_mask = (
    (sample_dates >= pd.Timestamp(CFG.test_start_date)) &
    (sample_dates <= pd.Timestamp(CFG.test_end_date))
)

X_test = X_all[test_mask.to_numpy()]
Y_test = Y_all[test_mask.to_numpy()]
D_test = D_current_all[test_mask.to_numpy()]
dates_test = sample_dates[test_mask].reset_index(drop=True)

print("X_test shape :", X_test.shape)
print("Y_test shape :", Y_test.shape)
print("D_test shape :", D_test.shape)
print("Test dates   :", dates_test.min().date(), "->", dates_test.max().date())

X_test shape : (1235, 100, 8, 8)
Y_test shape : (1235, 8, 8)
D_test shape : (1235, 8, 8)
Test dates   : 2021-01-04 -> 2025-12-02


In [54]:
import torch

# =========================================================
# APPLY CONSTANT TRAINED MODEL TO FULL TEST WINDOW
# =========================================================
cab_model.eval()

X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    out = cab_model(X_test_tensor)
    Y_model_sym = out["y_sym"]

# Move back to numpy
Y_model_sym = Y_model_sym.detach().cpu().numpy()

# PSD correction sample-by-sample using the helper logic
def numpy_enforce_psd(Y_batch: np.ndarray) -> np.ndarray:
    Y_psd_list = []
    for k in range(Y_batch.shape[0]):
        M = Y_batch[k]
        M = 0.5 * (M + M.T)
        eigvals, eigvecs = np.linalg.eigh(M)
        eigvals = np.clip(eigvals, 0.0, None)
        M_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
        Y_psd_list.append(M_psd)
    return np.asarray(Y_psd_list)

Y_model_psd = numpy_enforce_psd(Y_model_sym)

# Shrinkage: 0.8 * model + 0.2 * D_t
phi = 0.8
Y_model_shrunk = phi * Y_model_psd + (1 - phi) * D_test

print("Y_model_sym shape   :", Y_model_sym.shape)
print("Y_model_psd shape   :", Y_model_psd.shape)
print("Y_model_shrunk shape:", Y_model_shrunk.shape)

Y_model_sym shape   : (1235, 8, 8)
Y_model_psd shape   : (1235, 8, 8)
Y_model_shrunk shape: (1235, 8, 8)


In [55]:
# =========================================================
# FLATTEN MATRICES INTO DATAFRAMES
# =========================================================
cov_cols = [
    f"cov_{a}__{b}"
    for a in CFG.asset_order
    for b in CFG.asset_order
]

def matrix_batch_to_df(Y_batch: np.ndarray, dates: pd.Series, columns: list) -> pd.DataFrame:
    rows = []
    for k in range(Y_batch.shape[0]):
        row = {"Date": dates.iloc[k]}
        flat = Y_batch[k].reshape(-1)
        for col, val in zip(columns, flat):
            row[col] = val
        rows.append(row)
    return pd.DataFrame(rows)

df_realized = matrix_batch_to_df(Y_test, dates_test, cov_cols)
df_cab = matrix_batch_to_df(Y_model_psd, dates_test, cov_cols)
df_cab_shrunk = matrix_batch_to_df(Y_model_shrunk, dates_test, cov_cols)

print(df_realized.shape, df_cab.shape, df_cab_shrunk.shape)

(1235, 65) (1235, 65) (1235, 65)


In [59]:
for name, param in cab_model.named_parameters():
    print(name, param.shape)

cnn_block.conv3d.weight torch.Size([1, 1, 5, 5, 5])
cnn_block.conv3d.bias torch.Size([1])
bilstm_block.bilstm.weight_ih_l0 torch.Size([512, 64])
bilstm_block.bilstm.weight_hh_l0 torch.Size([512, 128])
bilstm_block.bilstm.bias_ih_l0 torch.Size([512])
bilstm_block.bilstm.bias_hh_l0 torch.Size([512])
bilstm_block.bilstm.weight_ih_l0_reverse torch.Size([512, 64])
bilstm_block.bilstm.weight_hh_l0_reverse torch.Size([512, 128])
bilstm_block.bilstm.bias_ih_l0_reverse torch.Size([512])
bilstm_block.bilstm.bias_hh_l0_reverse torch.Size([512])
bilstm_block.bilstm.weight_ih_l1 torch.Size([512, 256])
bilstm_block.bilstm.weight_hh_l1 torch.Size([512, 128])
bilstm_block.bilstm.bias_ih_l1 torch.Size([512])
bilstm_block.bilstm.bias_hh_l1 torch.Size([512])
bilstm_block.bilstm.weight_ih_l1_reverse torch.Size([512, 256])
bilstm_block.bilstm.weight_hh_l1_reverse torch.Size([512, 128])
bilstm_block.bilstm.bias_ih_l1_reverse torch.Size([512])
bilstm_block.bilstm.bias_hh_l1_reverse torch.Size([512])
attentio

In [60]:
fc_weight = cab_model.forecast_head.fc.weight.detach().cpu().numpy()
fc_bias   = cab_model.forecast_head.fc.bias.detach().cpu().numpy()

print("fc weight abs mean:", np.mean(np.abs(fc_weight)))
print("fc weight abs max :", np.max(np.abs(fc_weight)))
print("fc bias abs mean  :", np.mean(np.abs(fc_bias)))
print("fc bias abs max   :", np.max(np.abs(fc_bias)))

fc weight abs mean: 0.0002713967
fc weight abs max : 0.018165514
fc bias abs mean  : 0.012223232
fc bias abs max   : 0.03226573
